# Calculations

Count tables from the raw files, Potts fits with bootstrap intervals and RMSE, effective conformity per state, structured-state predictions, semantic fits with preference fields, binary endgame fit, numbers quoted in the thesis. Runs on `data/plot_sources/` 

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar

RAW = Path("../data/raw")
DATA = Path("../data/plot_sources")
OLD_GRID = [0.0, 0.1, 0.25, 0.4, 0.6, 0.8, 1.0]      # seven-point grid of the first measurement round

## Count tables

One row per state and reply with a count, from the raw files of notebooks 02 to 04 and 07 (skipped when `data/raw/` is empty, the tables are in the repository).

In [ ]:
def count_table(df, keys):
    return df.groupby(keys, dropna=False, sort=False).size().rename("n").reset_index()


if (RAW / "response_function_queries.csv").exists():
    raw = pd.read_csv(RAW / "response_function_queries.csv")
    raw["valid"] = raw["chosen_role"].notna()
    neutral = raw[raw["mode"] == "neutral"]
    count_table(neutral, ["model_label", "q", "n_display", "rel", "leader_share", "counts", "chosen_role"]) \
        .to_csv(DATA / "neutral_response_counts.csv", index=False)
    anchor = neutral[(neutral["q"] == 2) & (neutral["n_display"] == 49)]
    count_table(anchor, ["model_label", "q", "n_display", "rel", "leader_share", "shares_options",
                         "chosen_display", "valid"]).to_csv(DATA / "binary_anchor_counts.csv", index=False)
    structured = raw[raw["mode"] == "structured"].rename(columns={"set_name": "state"})
    count_table(structured, ["model_label", "state", "q", "n_display", "counts", "chosen_role"]) \
        .to_csv(DATA / "structured_response_counts.csv", index=False)

if (RAW / "semantic_queries.csv").exists():
    sem_raw = pd.read_csv(RAW / "semantic_queries.csv")
    sem_raw["valid"] = sem_raw["chosen_role"].notna()
    count_table(sem_raw, ["model_label", "set_name", "q", "n_display", "rel", "leader_option_idx",
                          "shares_options", "chosen_option_idx", "valid"]) \
        .to_csv(DATA / "semantic_response_counts.csv", index=False)

if (RAW / "endgame_queries.csv").exists():
    end_raw = pd.read_csv(RAW / "endgame_queries.csv")
    end_raw = end_raw[end_raw["chosen_role"].notna()]
    end = end_raw.groupby(["model_label", "N", "n_display", "n_lead", "n_other", "labels"], sort=False) \
        .agg(queries=("chosen_role", "size"), p_lead=("chosen_role", lambda r: (r == 0).mean())).reset_index()
    end["p_other"] = 1 - end["p_lead"]
    end.rename(columns={"model_label": "model", "n_display": "displayed"}).round(3) \
        .to_csv(DATA / "endgame_same_label_test.csv", index=False)

if (RAW / "trajectories.csv").exists():
    traj = pd.read_csv(RAW / "trajectories.csv")
    traj.drop(columns=["counts", "invalid_count"]).to_csv(DATA / "dynamics_trajectories.csv", index=False)
    pd.read_csv(RAW / "summary.csv").to_csv(DATA / "dynamics_summary.csv", index=False)

if (RAW / "abliterated_queries.csv").exists():
    abl = pd.read_csv(RAW / "abliterated_queries.csv")
    abl["valid"] = abl["chosen_role"].notna()
    count_table(abl, ["model_label", "block_id", "label_mode", "q", "N", "n_display", "rel", "leader_share",
                      "counts_role", "designated_leader_label", "chosen_role", "valid"]) \
        .to_csv(DATA / "abliterated_response_counts.csv", index=False)
    pd.read_csv(RAW / "abliterated_trajectories.csv").drop(columns=["counts", "invalid_count"]) \
        .to_csv(DATA / "abliterated_dynamics_trajectories.csv", index=False)
    pd.read_csv(RAW / "abliterated_summary.csv").to_csv(DATA / "abliterated_dynamics_summary.csv", index=False)

## Potts fit per block (model, q, N)

In [ ]:
counts = pd.read_csv(DATA / "neutral_response_counts.csv")
valid = counts[counts["chosen_role"].notna()].copy()
valid["chosen_role"] = valid["chosen_role"].astype(int)
print("queries:", counts["n"].sum(), " valid:", valid["n"].sum(), " rate: %.4f" % (valid["n"].sum() / counts["n"].sum()))


def softmax_rows(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def tally(block):
    # S: share vector per state (index 0 = leader), C: choice counts per state and role
    states = block.groupby(["rel", "counts"], sort=False).size().reset_index()[["rel", "counts"]]
    S = np.array([np.array(json.loads(c)) / sum(json.loads(c)) for c in states["counts"]])
    C = np.zeros((len(S), S.shape[1]))
    idx = {(r, c): i for i, (r, c) in enumerate(zip(states["rel"], states["counts"]))}
    for r, c, role, n in zip(block["rel"], block["counts"], block["chosen_role"], block["n"]):
        C[idx[(r, c)], role] += n
    return states, S, C


def nll(beta, S, C):
    # P(a|m) = exp(2 beta m_a) / sum_g exp(2 beta m_g)
    P = softmax_rows(2.0 * beta * S)
    return -(C * np.log(np.clip(P, 1e-300, None))).sum()


def fit_beta(S, C, bmax=500.0):
    return minimize_scalar(lambda b: nll(b, S, C), bounds=(0, bmax), method="bounded").x


def bootstrap(S, C, n_boot=200, seed=0):
    # multinomial resampling of the counts
    rng = np.random.default_rng(seed)
    n, p = int(C.sum()), C.flatten() / C.sum()
    vals = sorted(fit_beta(S, rng.multinomial(n, p).reshape(C.shape)) for _ in range(n_boot))
    return vals[int(0.025 * n_boot)], vals[int(0.975 * n_boot)]


def beta_c(q):
    return 1.0 if q == 2 else (q - 1) / (q - 2) * math.log(q - 1)

In [ ]:
rows = []
for (ml, q, nd), block in valid.groupby(["model_label", "q", "n_display"]):
    states, S, C = tally(block)
    beta = fit_beta(S, C)
    lo, hi = bootstrap(S, C)
    n_state = C.sum(axis=1)
    p_lead = C[:, 0] / n_state
    p_fit = softmax_rows(2.0 * beta * S)[:, 0]
    pos = states["rel"].values > 0
    # states where the leader is neither always nor never chosen; <= 2 of them: beta poorly determined (flag)
    informative = int(((p_lead >= 0.01) & (p_lead <= 0.99) & pos).sum())
    rows.append({"model_label": ml, "q": q, "n_display": nd, "n_valid": int(C.sum()), "design_points": len(S),
                 "beta_m1": beta, "beta_ci_lo": lo, "beta_ci_hi": hi,
                 "p_leader_smallest_lead": p_lead[pos][np.argmin(S[pos, 0])],
                 "n_informative_points": informative, "beta_is_lower_bound": informative <= 2,
                 "beta_c": beta_c(q), "beta_over_beta_c": beta / beta_c(q),
                 "rmse": math.sqrt((n_state * (p_lead - p_fit) ** 2).sum() / n_state.sum())})   # query-weighted
fits = pd.DataFrame(rows)
fits.to_csv(DATA / "neutral_parameters.csv", index=False)
print(fits[["model_label", "q", "n_display", "beta_m1", "beta_ci_lo", "beta_ci_hi", "beta_is_lower_bound", "beta_c", "rmse"]]
      .round(3).to_string(index=False))

In [ ]:
# RMSE per model, pooled over its blocks (chapter 4)
pooled = fits.groupby("model_label").apply(lambda g: math.sqrt((g["n_valid"] * g["rmse"] ** 2).sum() / g["n_valid"].sum()),
                                           include_groups=False)
print(pooled.round(3))
print("poorly determined blocks:", fits["beta_is_lower_bound"].sum())
print("blocks below beta_c:")
print(fits.loc[fits["beta_m1"] < fits["beta_c"], ["model_label", "q", "n_display", "beta_m1", "beta_c"]].round(2).to_string(index=False))

## Effective conformity per state

In [ ]:
def beta_eff_row(k, n, q, s):
    # invert the softmax at one state, Haldane-Anscombe correction, Wald interval on the logit scale
    r = (1.0 - s) / (q - 1)
    p = (k + 0.5) / (n + 1.0)
    logit = math.log(p / (1 - p)) + math.log(q - 1)
    se = math.sqrt(1.0 / (k + 0.5) + 1.0 / (n - k + 0.5))
    return {"rest_share": r, "lead": s - r, "beta_eff": logit / (2 * (s - r)),
            "beta_eff_ci_lo": (logit - 1.96 * se) / (2 * (s - r)),
            "beta_eff_ci_hi": (logit + 1.96 * se) / (2 * (s - r)),
            "at_boundary": (k == 0) or (k == n)}


rows = []
for (ml, q, nd, rel), g in valid[valid["rel"] > 0].groupby(["model_label", "q", "n_display", "rel"]):
    n = int(g["n"].sum())
    k = int(g.loc[g["chosen_role"] == 0, "n"].sum())
    s = float(g["leader_share"].iloc[0])
    row = {"model_label": ml, "q": q, "n_display": nd, "rel": rel, "leader_share": s,
           "n_valid": n, "k_leader": k, "p_leader": k / n}
    row.update(beta_eff_row(k, n, q, s))
    rows.append(row)
beta_eff = pd.DataFrame(rows).sort_values(["model_label", "q", "n_display", "leader_share"])
beta_eff.to_csv(DATA / "effective_parameters.csv", index=False)

In [ ]:
# smallest resolved lead per block at N = 200 and the beta_eff profiles
inside = beta_eff[~beta_eff["at_boundary"]]
smallest = inside.sort_values("lead").groupby(["model_label", "q", "n_display"]).first()
print(smallest.loc[(slice(None), slice(None), 199), ["lead", "beta_eff", "beta_eff_ci_lo", "beta_eff_ci_hi"]].round(1).to_string())

for ml, q in [("qwen25_32b_it", 50), ("qwen25_32b_it", 2), ("qwen25_7b_it", 50), ("qwen25_7b_it", 100)]:
    prof = inside[(inside["model_label"] == ml) & (inside["q"] == q) & (inside["n_display"] == 199)]
    pooled_beta = fits.loc[(fits["model_label"] == ml) & (fits["q"] == q) & (fits["n_display"] == 199), "beta_m1"].iloc[0]
    print(f"{ml} q={q} N=200: beta_eff from {prof['beta_eff'].iloc[0]:.1f} to {prof['beta_eff'].iloc[-1]:.1f}, pooled {pooled_beta:.1f}")

# the 400:399 binary state at N = 800
one_agent = valid[(valid["q"] == 2) & (valid["n_display"] == 799) & (valid["rel"] == 0.1)]
print(one_agent.groupby("model_label").apply(lambda g: g.loc[g["chosen_role"] == 0, "n"].sum() / g["n"].sum(),
                                             include_groups=False).round(3))

## Structured states: prediction from the slice fit

In [ ]:
structured = pd.read_csv(DATA / "structured_response_counts.csv")
structured = structured[structured["chosen_role"].notna()].copy()
structured["chosen_role"] = structured["chosen_role"].astype(int)
old = valid[valid["rel"].isin(OLD_GRID) & (valid["n_display"] == 49)]     # measured together with the first round

rows = []
for (ml, state), g in structured.groupby(["model_label", "state"]):
    q = int(g["q"].iloc[0])
    _, S, C = tally(old[(old["model_label"] == ml) & (old["q"] == q)])
    beta = fit_beta(S, C)
    shares = np.array(json.loads(g["counts"].iloc[0]), dtype=float) / 49
    n = int(g["n"].sum())
    k = int(g.loc[g["chosen_role"] == 0, "n"].sum())
    rows.append({"model_label": ml, "state": state, "q": q, "n": n, "obs_P_leader": k / n,
                 "M1_P_leader": softmax_rows(2.0 * beta * shares[None, :])[0, 0]})
struct_val = pd.DataFrame(rows)
struct_val.to_csv(DATA / "structured_validation.csv", index=False)

four = struct_val[struct_val["model_label"].isin(["gemma4_E4B_it", "gemma4_31B_dense", "qwen25_7b_it", "qwen25_32b_it"])]
diff = four["obs_P_leader"] - four["M1_P_leader"]
print("mean absolute error: %.3f, within 0.10: %d of %d" % (diff.abs().mean(), (diff.abs() < 0.10).sum(), len(four)))
print("strong runner-up, mean overprediction: %.3f" % -diff[four["state"].str.contains("strong_runner")].mean())
print(four.round(3).to_string(index=False))

## Semantic sets: conformity with preference fields

In [ ]:
sem = pd.read_csv(DATA / "semantic_response_counts.csv")
sem = sem[sem["valid"] & sem["chosen_option_idx"].notna()].copy()
sem["chosen_option_idx"] = sem["chosen_option_idx"].astype(int)

OPTIONS = {
    "energy_q2_anchor": ["renewable energy", "fossil fuels"],
    "energy_q3": ["renewable energy", "nuclear power", "fossil fuels"],
    "justice_q3": ["rehabilitative justice", "punitive justice", "restorative justice"],
    "speech_q3": ["unrestricted free speech", "regulated speech", "harm-reduction moderation"],
    "ai_governance_q4": ["open-source AI", "corporate self-regulation", "government regulation", "international AI treaty"],
    "beverage_q5": ["coffee", "tea", "water", "juice", "soda"],
    "political_ideology_q6": ["liberalism", "conservatism", "socialism", "libertarianism", "green politics", "nationalism"],
    "pets_q7": ["cats", "dogs", "birds", "fish", "rabbits", "hamsters", "reptiles"],
    "policy_priorities_q10": ["healthcare", "education", "climate policy", "economic growth", "public safety",
                              "housing", "immigration", "digital rights", "national defense", "social welfare"],
    "ai_values_q10": ["helpfulness", "honesty", "harmlessness", "privacy", "fairness", "transparency",
                      "autonomy", "robustness", "accountability", "efficiency"],
}


def sem_tally(g):
    # shares indexed by option (the leader rotates), one row per distinct state
    keys, S, rows = {}, [], []
    for sh, ch in zip(g["shares_options"], g["chosen_option_idx"]):
        k = tuple(round(x, 10) for x in json.loads(sh))
        if k not in keys:
            keys[k] = len(S)
            S.append(np.array(json.loads(sh)))
        rows.append((keys[k], ch))
    S = np.array(S)
    C = np.zeros((len(S), S.shape[1]))
    for (si, ch), n in zip(rows, g["n"]):
        C[si, ch] += n
    return S, C


def sem_nll(p, S, C):
    # P(a|m) = exp(2 beta m_a + h_a) / sum, h of the last option fixed at zero
    beta, h = p[0], np.append(p[1:], 0.0)
    P = softmax_rows(2.0 * beta * S + h)
    return -(C * np.log(np.clip(P, 1e-300, None))).sum()


def sem_fit(S, C, q, bmax=250.0, hmax=15.0):
    # |h| <= 15 keeps the fit finite when one option is chosen at almost every state
    x0 = np.concatenate([[3.0], np.zeros(q - 1)])
    r = minimize(sem_nll, x0=x0, args=(S, C), bounds=[(0, bmax)] + [(-hmax, hmax)] * (q - 1), method="L-BFGS-B")
    return r.x[0], np.append(r.x[1:], 0.0)

In [ ]:
# neutral reference at the same q and N = 50, refitted on the seven-point grid of the semantic design
neutral49 = fits[fits["n_display"] == 49].set_index(["model_label", "q"])["beta_m1"]
grid_matched = {}
for (ml, q), g in old.groupby(["model_label", "q"]):
    _, S, C = tally(g)
    grid_matched[(ml, q)] = fit_beta(S, C, bmax=300.0)

rng = np.random.default_rng(0)
records, fields = [], []
for (ml, set_name), g in sem.groupby(["model_label", "set_name"]):
    q = int(g["q"].iloc[0])
    S, C = sem_tally(g)
    beta, h = sem_fit(S, C, q)
    n, p = int(C.sum()), C.flatten() / C.sum()
    boots = [sem_fit(S, rng.multinomial(n, p).reshape(C.shape), q)[0] for _ in range(200)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    h_range = h.max() - h.min()
    records.append({"model_label": ml, "set_name": set_name, "q": q, "n_queries": n,
                    "beta": beta, "beta_ci_lo": lo, "beta_ci_hi": hi, "h_range": h_range,
                    "overturn_lead": h_range / (2 * beta) if beta > 0 else np.inf,     # lead that cancels the strongest preference
                    "h_at_bound": bool(np.any(np.abs(h[:-1]) >= 14.99)),
                    "neutral_beta_q_N50": neutral49.get((ml, q), np.nan),
                    "neutral_beta_gridmatched": grid_matched.get((ml, q), np.nan)})
    for option, hv, hc in zip(OPTIONS[set_name], h, h - h.mean()):
        fields.append({"model_label": ml, "set_name": set_name, "q": q, "option": option, "h": hv, "h_centred": hc})
    print(f"{ml:18s} {set_name:24s} beta={beta:6.2f} [{lo:5.2f}, {hi:5.2f}]")

sem_fits = pd.DataFrame(records)
sem_fits.to_csv(DATA / "semantic_parameters.csv", index=False)
sem_h = pd.DataFrame(fields)
sem_h.to_csv(DATA / "semantic_fields.csv", index=False)

## Binary endgame fit (Qwen 2.5 7B, N = 100, section 4.3)

In [ ]:
end = pd.read_csv(DATA / "endgame_same_label_test.csv")


def fit_tanh(k, n, m):
    # P(m) = (1 + tanh(beta m)) / 2, maximum likelihood via the score equatio
    k = np.asarray(k)
    lower, upper = np.zeros(k.shape[:-1]), np.full(k.shape[:-1], 100.0)
    for _ in range(80):
        beta = (lower + upper) / 2
        p = 0.5 * (1 + np.tanh(beta[..., None] * m))
        score = np.sum(m * (k - n * p), axis=-1)
        lower = np.where(score > 0, beta, lower)
        upper = np.where(score > 0, upper, beta)
    return (lower + upper) / 2


for labels in ["q50_pool", "aa_ab"]:
    g = end[end["labels"] == labels].sort_values("n_lead")
    n = g["queries"].to_numpy()
    k = np.rint(n * g["p_lead"]).astype(int)
    m = ((g["n_lead"] - g["n_other"]) / g["displayed"]).to_numpy()
    beta = float(fit_tanh(k, n, m))
    boots = np.random.default_rng(20260906).binomial(n, k / n, size=(20000, len(n)))     # binomial bootstrap per state
    lo, hi = np.quantile(fit_tanh(boots, n, m), [0.025, 0.975])
    print(f"{labels:9s} beta_bin = {beta:.2f} [{lo:.2f}, {hi:.2f}]   beta_c = 1, beta_t(100) = {0.5 * math.log(100):.2f}")

q7 = fits[(fits["model_label"] == "qwen25_7b_it") & (fits["n_display"] == 99)].set_index("q")["beta_m1"]
print("neutral blocks, Qwen 2.5 7B N=100: q=2 %.2f, q=50 %.2f" % (q7[2], q7[50]))

## Numbers quoted in Appendix B

In [ ]:
sem_counts = pd.read_csv(DATA / "semantic_response_counts.csv")
five = ["energy_q2_anchor", "speech_q3", "ai_governance_q4", "political_ideology_q6", "policy_priorities_q10"]
f = sem_counts[sem_counts["set_name"].isin(five)]
print("five sets: queries", f["n"].sum(), " valid", f.loc[f["valid"], "n"].sum(), " rate %.3f" % (f.loc[f["valid"], "n"].sum() / f["n"].sum()))


def leader_curve(ml, set_name):
    # P(option | option leads with share s), observed frequency and biased-Potts prediction per option
    g = sem[(sem["model_label"] == ml) & (sem["set_name"] == set_name)]
    fit = sem_fits[(sem_fits["model_label"] == ml) & (sem_fits["set_name"] == set_name)].iloc[0]
    h = sem_h[(sem_h["model_label"] == ml) & (sem_h["set_name"] == set_name)]["h"].to_numpy()
    rows = []
    for (idx, sh), gg in g.groupby(["leader_option_idx", "shares_options"]):
        shares = np.array(json.loads(sh))
        rows.append({"option": OPTIONS[set_name][idx], "s": shares[idx], "n": gg["n"].sum(),
                     "observed": gg.loc[gg["chosen_option_idx"] == idx, "n"].sum() / gg["n"].sum(),
                     "fitted": softmax_rows(2.0 * fit["beta"] * shares[None, :] + h)[0, idx]})
    return pd.DataFrame(rows)


# energy pair at the largest fossil-fuel majority 
for ml in ["qwen25_7b_it", "gemma4_31B_dense", "qwen25_32b_it"]:
    c = leader_curve(ml, "energy_q2_anchor")
    r = c[(c["option"] == "fossil fuels") & (c["s"] == c["s"].max())].iloc[0]
    print(f"{ml}: renewable chosen against 48:1 fossil fuels in {round((1 - r['observed']) * r['n'])} of {r['n']} replies")
print("Qwen 2.5 32B energy beta: %.2f" % sem_fits[(sem_fits["model_label"] == "qwen25_32b_it") & (sem_fits["set_name"] == "energy_q2_anchor")]["beta"].iloc[0])

In [ ]:
# largest lead per option, the four multi-option sets, Gemma and Qwen models
four_models = ["gemma4_E4B_it", "gemma4_31B_dense", "qwen25_7b_it", "qwen25_32b_it"]
top = []
for ml in four_models:
    for set_name in five[1:]:
        c = leader_curve(ml, set_name)
        top.append(c[c["s"] == c["s"].max()].assign(model_label=ml, set_name=set_name))
top = pd.concat(top)
print("option-model combinations with P(leader) >= 0.90 at the largest lead:", (top["observed"] >= 0.90).sum(), "of", len(top))
print(top[top["set_name"] == "political_ideology_q6"].pivot(index="option", columns="model_label", values="observed").round(2))
print(top[top["set_name"] == "ai_governance_q4"].pivot(index="option", columns="model_label", values="observed").round(2))
c = leader_curve("qwen25_32b_it", "political_ideology_q6")
print(c[c["s"] == c["s"].max()].round(3).to_string(index=False))

In [ ]:
# exploratory biased-Potts fit: query-weighted RMSE over the 
rmse = {}
for ml in sem_fits["model_label"].unique():
    for set_name in five:
        c = leader_curve(ml, set_name)
        rmse[(ml, set_name)] = math.sqrt((c["n"] * (c["observed"] - c["fitted"]) ** 2).sum() / c["n"].sum())
print(pd.Series(rmse).unstack().round(3))
c = leader_curve("llama31_8b_it", "political_ideology_q6")
print(c[(c["option"] == "conservatism") & (c["s"] == c["s"].max())].round(2).to_string(index=False))

## Numbers quoted in Appendix C

In [ ]:
abl_summary = pd.read_csv(DATA / "abliterated_dynamics_summary.csv")
abl_traj = pd.read_csv(DATA / "abliterated_dynamics_trajectories.csv")
abl_counts = pd.read_csv(DATA / "abliterated_response_counts.csv")

print(abl_summary.groupby(["condition_id", "model_label"]).agg(done=("event_observed", "sum"),
                                                              mean_T=("consensus_time_sweeps", "mean")).round(2))
print(abl_summary[abl_summary["condition_id"] == "political_ideology_q6_N50"][["model_label", "run", "winner_label"]].to_string(index=False))

# stalled N = 200 runs: first time with two active opinions
reg = abl_traj[(abl_traj["record_type"] == "regular") & (abl_traj["N"] == 200)]
first_two = reg[reg["n_active_opinions"] <= 2].groupby(["model_label", "condition_id", "run"])["time_sweeps"].min()
print(first_two.groupby(level=[0, 1]).max().round(1))

# ideology block: adoption of each ideology in the leader role
ide = abl_counts[(abl_counts["block_id"] == "political_ideology_q6_N50") & abl_counts["valid"]]
ide = ide.groupby(["model_label", "designated_leader_label", "leader_share"]).apply(
    lambda g: pd.Series({"n": g["n"].sum(), "k": g.loc[g["chosen_role"] == 0, "n"].sum()}), include_groups=False).reset_index()
for option, share in [("socialism", 44 / 49), ("libertarianism", 16 / 49), ("green politics", 18 / 49)]:
    print(option, ide[(ide["designated_leader_label"] == option) & ((ide["leader_share"] - share).abs() < 1e-9)]
          [["model_label", "n", "k"]].to_string(index=False))

## Simulation outcomes per condition

In [ ]:
summ = pd.read_csv(DATA / "dynamics_summary.csv")
table = summ.groupby(["model_label", "q", "N"]).agg(runs=("run", "size"), consensus=("event_observed", "sum"),
                                                     median_T=("consensus_time_sweeps", "median"),
                                                     final_share=("leader_share", "mean")).round(2)
print(table.to_string())